# Homework: milestone 1, it loads, it cleans, and you can prove it

Today's first run was deliberately rough. Milestone 1 makes the first two
stages of the pipeline **honest**: you know exactly which rows went in,
which came out and why, and a set of checks says so every time it runs.

Milestone 1 is done when:

* `project/pipeline.py` loads your real data and cleans it, printing one
  line per cleaning step with the rows before and after
* a `find_problems` function finds nothing wrong with the cleaned data
* `analyse` produces your **must** output, and it is checked two ways
* `python project/pipeline.py` runs start to finish and writes the output
  to `project/output/`
* it is committed and pushed, and the *decisions* part of your worksheet
  says what you decided and why

Work it out here first, then move it into `project/pipeline.py` (part 4).
Notebooks are for working things out; the script is where it has to run.

In [ ]:
import os
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

In [ ]:
DATA_PATH = Path("project/data/YOUR-FILE-NAME.csv")      # TODO: your file

# TODO: raw = pd.read_csv(DATA_PATH, ...)   with the arguments you found today

## Part 1: cleaning, one logged step at a time

Write each cleaning step as its own small function that takes a DataFrame
and **returns** a new one. Then run them through `logged`, below, which
prints how many rows each step kept. Do not print inside the steps; the
outer layer prints, as always.

The usual steps, taken from session 9. Use the ones your data needs, in
whatever order makes sense, and no others:

* keep only the columns your question uses (and rename awkward ones)
* tidy text: `.str.strip()`, `.str.lower()`
* turn text into numbers or dates: `pd.to_numeric(..., errors="coerce")`,
  `pd.to_datetime(..., format=...)`
* drop exact duplicates
* drop, or fill, the rows with missing values in the columns you need
* keep only the date range your question is about

In [ ]:
def logged(df, *steps):
    """Run each step on df in turn, print rows before and after each, return the result."""
    # *steps means "any number of arguments, collected into a tuple".
    # Each one is a function, and step.__name__ is that function's name as text.
    print(f"{'start':32} {len(df):>7,} rows")
    for step in steps:
        before = len(df)
        df = step(df)
        dropped = before - len(df)
        print(f"{step.__name__:32} {len(df):>7,} rows   ({dropped:,} dropped)")
    return df


# TODO: your cleaning steps, one function each, for example
# def keep_needed_columns(df):
#     """..."""
#     return df[[...]].copy()


# TODO: clean = logged(raw, keep_needed_columns, ...)

Two things to notice in your log:

* every row that disappears is accounted for, step by step. If someone in
  session 12 asks "why do you have fewer rows than the spreadsheet?", the
  log is the answer.
* a step that drops nothing is still worth keeping. It is a check that
  something you were worried about did not happen.

For every step that dropped or changed rows, add a line to the
*decisions* part of your worksheet: what you did, how many rows it
touched, and why that was the right call for your question.

## Part 2: `find_problems`

An `assert` stops at the first failure. When you are checking data, you
want to see **every** problem at once. So write a function that returns a
list of problems, as sentences, and an empty list when all is well.

Check whatever your question depends on. Good starting points:

* the columns you need are all there
* none of them has missing values
* the numbers are numbers (`pd.api.types.is_numeric_dtype(df[column])`)
  and within a plausible range
* the dates cover what your question needs
* no duplicates of whatever identifies a row

The skeleton below does the first check, so you can see the shape.

In [ ]:
NEEDED = ["TODO", "your", "columns"]


def find_problems(df):
    """Return a list of problems with the cleaned data, as sentences. Empty means fine."""
    problems = []

    missing_columns = [c for c in NEEDED if c not in df.columns]
    if missing_columns:
        problems.append(f"missing columns: {missing_columns}")
        return problems              # the other checks need the columns, so stop here

    # TODO: more checks, each one appending a sentence to problems

    return problems

Run it on your raw data first. It should find something, or your checks
are not looking hard enough. Then run it on your cleaned data, where it
should find nothing.

In [ ]:
# TODO: print(find_problems(raw))
# TODO: problems = find_problems(clean)
#       assert not problems, problems

## Part 3: the must output, checked two ways

Write `analyse(df)` so it returns your must output. Then check one number
in it by a **different route**: the parts add up to the whole, or one cell
worked out with a filter instead of a `groupby`. Notebook 02 (solved)
did both.

In [ ]:
# TODO: def analyse(df):


# TODO: result = analyse(clean)
# TODO: two asserts, two routes to the same number

## Part 4: into `project/pipeline.py`

1. Copy your step functions, `logged`, `find_problems` and `analyse` into
   `project/pipeline.py`. Your `clean(df)` there becomes one line:
   `return logged(df, step_one, step_two, ...)`. Keep the session 10 habit
   of listing the decisions in `clean`'s docstring, too.
2. In `main()`, straight after `data = clean(raw)`, add:

   ```python
   problems = find_problems(data)
   assert not problems, problems
   ```

   Now the pipeline refuses to write an output from data it does not
   trust. That is the most valuable two lines in the project.
3. Give `check` a real job again, if you commented it out last time: read
   the CSV back and compare one total in it with the same total from the
   cleaned data. Your two-routes `assert` from part 3 is a good candidate.
4. Run `python project/pipeline.py` from the repository root. The log
   prints, the checks pass, `summary.csv` appears in `project/output/`.
5. Open the CSV. Same numbers as this notebook?
6. Look at `git status`, then commit and push, with a message that says
   what milestone 1 was. Not your data, if it is private: see
   `homework/README.md`.

**Saturday:** one line, `Milestone 1 done` or `Milestone 1 not done`, and
if not, which part stopped you.

## If you have time left

Start milestone 2: the **should** item from your worksheet, which for most
projects is the chart. One chart, one question, the answer in the title,
labels with `:.1f`, saved as a PNG into `project/output/` by a `save`
step. Session 9 has all the pieces.